# Suite VIS — Visitors protocol conformance

Expressions' builders implement `Visitors.OfObject` and its parts (the protocols `FromPlain` builds through), and
expression data implements `Visitable`. Each must implement its protocol completely, with matching arity, on the class
and on live instances. (mbse-schemas' own VIS suite covers its implementations.)

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Expressions.Framework import Terms as F
from mbse.Schemas.Framework import Visitors as V
from support import conformance_problems, instance_problems

## VIS-01 · Every builder, builder part and kind of data conforms to its protocol

In [ ]:
implementations = [
    (E.OfLiteral.Builder, V.OfObject), (E.OfOperation.Builder, V.OfObject), (E.OfVariable.Builder, V.OfObject),
    (E.OfLet.Builder, V.OfObject), (F._Adjacency, V.OfAdjacency), (F._Argument, V.OfEntry), (F._Link, V.OfLink),
    (F._Field, V.OfProperty), (F._Field, V.OfAny), (F._Field, V.OfNative),
    (E.OfLiteral.Data, V.Visitable), (E.OfOperation.Data, V.Visitable), (E.OfVariable.Data, V.Visitable),
    (E.OfLet.Data, V.Visitable),
]
problems = [p for impl, proto in implementations for p in conformance_problems(impl, proto)]
assert problems == [], problems

## VIS-02 · Live instances conform too

In [ ]:
found = [(E.OfLiteral.Builder(), V.OfObject), (E.OfOperation.Builder(), V.OfObject), (E.OfVariable.Builder(), V.OfObject),
         (E.OfLet.Builder(), V.OfObject), (E.OfLiteral.resolve(1), V.Visitable), (E.OfOperation.Data("f"), V.Visitable),
         (E.OfVariable.Data("x"), V.Visitable), (E.OfLet.Data("x"), V.Visitable)]
builder = E.OfOperation.Builder()
builder.property("name", lambda p: (found.append((p, V.OfProperty)), p.value(
    lambda a: (found.append((a, V.OfAny)), a.as_native(lambda n: found.append((n, V.OfNative)))))))
builder.adjacency("arguments", lambda a: (found.append((a, V.OfAdjacency)), a.add(
    lambda e: (found.append((e, V.OfEntry)), e.link("argument", lambda k: found.append((k, V.OfLink)))))))
problems = [p for instance, protocol in found for p in instance_problems(instance, protocol)]
assert problems == [], problems